# Getting Started with SurviveX

This notebook demonstrates the key features of **survivex**, a GPU-accelerated survival analysis library for Python.

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TaninZeraati/survivex/main?labpath=examples%2Fgetting_started.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# SurviveX imports
from survivex.models import (
    KaplanMeierEstimator,
    NelsonAalenEstimator,
    CoxPHModel,
    WeibullAFTFitter,
    RandomSurvivalForest,
)
from survivex.datasets import load_from_lifelines

print("SurviveX loaded successfully!")

## 1. Load Example Data

We'll use the Rossi recidivism dataset, which tracks 432 convicts for rearrest over one year.

In [ ]:
# Load the Rossi dataset
rossi = load_from_lifelines("rossi")
print(f"Dataset shape: {rossi.shape}")
print(f"Columns: {list(rossi.columns)}")
rossi.head()

In [ ]:
# Extract time, event, and covariates
T = rossi['week'].values.astype(float)  # Time to event
E = rossi['arrest'].values.astype(float)  # Event indicator (1=arrested, 0=censored)
covariates = ['fin', 'age', 'race', 'wexp', 'mar', 'paro', 'prio']
X = rossi[covariates].values.astype(float)

print(f"n = {len(T)} subjects")
print(f"Events = {int(E.sum())} arrests")
print(f"Censored = {int(len(E) - E.sum())}")

## 2. Kaplan-Meier Survival Curve

The Kaplan-Meier estimator provides a non-parametric estimate of the survival function.

In [ ]:
# Fit Kaplan-Meier estimator
km = KaplanMeierEstimator()
km.fit(T, E)

# Plot survival curve
fig, ax = plt.subplots(figsize=(10, 6))
times = km.timeline_.numpy()
survival = km.survival_function_.numpy()
ci_lower = km.confidence_interval_lower_.numpy()
ci_upper = km.confidence_interval_upper_.numpy()

ax.step(times, survival, where='post', label='Survival function', linewidth=2)
ax.fill_between(times, ci_lower, ci_upper, alpha=0.3, step='post', label='95% CI')
ax.set_xlabel('Time (weeks)', fontsize=12)
ax.set_ylabel('Survival Probability', fontsize=12)
ax.set_title('Kaplan-Meier Survival Curve (Rossi Dataset)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Median survival time: {km.median_survival_time_:.1f} weeks")

## 3. Cox Proportional Hazards Model

The Cox PH model estimates the effect of covariates on the hazard rate.

In [ ]:
# Fit Cox PH model
cox = CoxPHModel(tie_method='efron')
cox.fit(X, T, E)

# Display results
print("Cox Proportional Hazards Results")
print("=" * 60)
print(f"{'Covariate':<12} {'Coef':>10} {'HR':>10} {'SE':>10} {'p-value':>10}")
print("-" * 60)

from scipy.stats import norm
for i, name in enumerate(covariates):
    coef = cox.coefficients_[i]
    hr = np.exp(coef)
    se = cox.standard_errors_[i]
    z = coef / se
    p = 2 * (1 - norm.cdf(abs(z)))
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f"{name:<12} {coef:>10.4f} {hr:>10.4f} {se:>10.4f} {p:>10.4f} {sig}")

print("-" * 60)
print(f"Concordance Index: {cox.concordance_index_:.4f}")

## 4. Weibull AFT Model

The Weibull Accelerated Failure Time model assumes a parametric form for the survival distribution.

In [ ]:
# Fit Weibull AFT
weibull = WeibullAFTFitter()
weibull.fit(X, T, E)

print(f"Weibull shape parameter (rho): {weibull.rho_:.4f}")
print(f"Weibull scale parameter (lambda): {weibull.lambda_:.4f}")

# Predict survival for a specific covariate profile
X_mean = X.mean(axis=0).reshape(1, -1)
times_pred = np.array([10, 20, 30, 40, 50])
surv_pred = weibull.predict_survival_function(X_mean, times=times_pred)

print(f"\nPredicted survival at mean covariate values:")
for t, s in zip(times_pred, surv_pred):
    print(f"  S({t:2.0f} weeks) = {s:.4f}")

## 5. Random Survival Forest

Machine learning approach using an ensemble of survival trees.

In [ ]:
# Train/test split
np.random.seed(42)
n = len(T)
idx = np.random.permutation(n)
train_idx, test_idx = idx[:int(0.7*n)], idx[int(0.7*n):]

X_train, X_test = X[train_idx], X[test_idx]
T_train, T_test = T[train_idx], T[test_idx]
E_train, E_test = E[train_idx], E[test_idx]

# Fit Random Survival Forest
rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, random_state=42)
rsf.fit(X_train, T_train, E_train)

# Evaluate
c_index = rsf.score(X_test, T_test, E_test)
print(f"Random Survival Forest C-index (test): {c_index:.4f}")
print(f"OOB C-index: {rsf.oob_score_:.4f}")

# Feature importance
print(f"\nFeature Importance:")
importance = rsf.feature_importances_
for name, imp in sorted(zip(covariates, importance), key=lambda x: -x[1]):
    print(f"  {name}: {imp:.4f}")

## 6. Comparison with lifelines

Verify that survivex matches the reference implementation.

In [ ]:
from lifelines import CoxPHFitter

# Fit lifelines Cox PH
df = rossi[covariates + ['week', 'arrest']].copy()
cox_ll = CoxPHFitter(penalizer=0.0)
cox_ll.fit(df, duration_col='week', event_col='arrest')

# Compare coefficients
print("Coefficient Comparison: SurviveX vs lifelines")
print("=" * 50)
print(f"{'Covariate':<12} {'SurviveX':>12} {'lifelines':>12} {'Diff':>12}")
print("-" * 50)

max_diff = 0
for i, name in enumerate(covariates):
    sx_coef = cox.coefficients_[i]
    ll_coef = cox_ll.params_[name]
    diff = abs(sx_coef - ll_coef)
    max_diff = max(max_diff, diff)
    print(f"{name:<12} {sx_coef:>12.6f} {ll_coef:>12.6f} {diff:>12.2e}")

print("-" * 50)
print(f"Maximum coefficient difference: {max_diff:.2e}")
print(f"Validation: {'PASS' if max_diff < 1e-6 else 'FAIL'}")

## Summary

This notebook demonstrated:

1. **Kaplan-Meier estimator** - Non-parametric survival curve estimation
2. **Cox Proportional Hazards** - Semi-parametric regression with hazard ratios
3. **Weibull AFT** - Parametric survival model
4. **Random Survival Forest** - Machine learning approach
5. **Validation** - Coefficients match lifelines to machine precision

For more models (competing risks, multi-state, recurrent events, frailty), see the [full documentation](https://github.com/TaninZeraati/survivex).